# P11, le portefeuille d'avenir

La règle de P11 est écrite dans la section VII de `research/portefeuilles.md`, commitée avant ce calcul. **P11 est construit en connaissance des résultats des étapes 3 à 5 : son historique ci-dessous n'est pas une preuve**, seulement la vérification que la règle fait ce qu'elle annonce. Son test réel est le protocole prospectif écrit dans la même section.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src import risque
from src import couverture as cv
from src.collecter_couverture import verifier
from src.empreintes import empreinte

SORTIE = RACINE / "data" / "processed"
BRUT = RACINE / "data" / "raw"
verifier(RACINE)

valeurs = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)
dates = valeurs.index
comparaisons = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)
taux = pd.read_csv(BRUT / "taux_sans_risque.csv", index_col=0)["taux_annuel_pct"].reindex(dates).ffill()
sans_risque = (1 + taux / 100) ** (1 / 252) - 1
r_spy = comparaisons["SPY"].pct_change()
r_inter = pd.read_csv(BRUT / "couverture" / "VFITX.csv", index_col=0)["Adj Close"].pct_change().reindex(dates)

MAILLONS = {"P4": "acheteurs", "P5": "vendeurs", "P6": "electricite", "P7": "equipement",
            "P8": "immobilier", "P9": "amont des puces", "P10": "matieres et energie"}
PUCES = ["vendeurs", "amont des puces"]
ACTIONS, OBLIGATIONS, LIQUIDITES = 0.60, 0.20, 0.20

appartenance = pd.read_csv(SORTIE / "appartenance.csv")
p1 = set(appartenance[appartenance.portefeuille == "P1"].cik)
maillons = set().union(*[set(appartenance[appartenance.portefeuille == pf].cik) for pf in MAILLONS])
recouvrement = sum(len(set(appartenance[appartenance.portefeuille == a].cik) & set(appartenance[appartenance.portefeuille == b].cik))
                   for a in MAILLONS for b in MAILLONS if a < b)
print("entreprises de P1 :", len(p1), "| couvertes par les sept maillons :", len(p1 & maillons),
      "| hors P1 :", len(maillons - p1), "| recouvrements entre maillons :", recouvrement)

entreprises de P1 : 134 | couvertes par les sept maillons : 134 | hors P1 : 0 | recouvrements entre maillons : 0


## 1. Construction

In [2]:
poches = pd.DataFrame({nom: valeurs[f"{pf}_reeq"].pct_change() for pf, nom in MAILLONS.items()})
poches["obligations"] = r_inter
poches["liquidites"] = sans_risque

cibles, historique = {}, []
for jour in sorted(cv.premieres_seances_annee(dates)):
    passe = poches.loc[:jour, list(MAILLONS.values())].iloc[1:-1]
    w_actions = (cv.poids_inverse_volatilite(passe) if len(passe) >= 63
                 else pd.Series(1 / len(MAILLONS), index=list(MAILLONS.values())))
    w = pd.concat([ACTIONS * w_actions, pd.Series({"obligations": OBLIGATIONS, "liquidites": LIQUIDITES})])
    cibles[jour] = w
    historique.append(w.rename(jour))
historique = pd.DataFrame(historique)
p11, frais = cv.melange_multiple(poches, cibles)
print("frais de reequilibrage entre poches : %.3f pour 100 investis" % frais)
print()
print("poids cibles de la poche d'actions, en part des actions :")
print((historique[list(MAILLONS.values())] / ACTIONS).iloc[[0, 1, 5, 10, 15, 20, 25, -1]].round(3).to_string())
print()
print("part cible des puces dans les actions : min %.1f %%, max %.1f %%, derniere %.1f %%"
      % tuple(100 * x for x in [(historique[PUCES].sum(axis=1) / ACTIONS).min(),
                                (historique[PUCES].sum(axis=1) / ACTIONS).max(),
                                (historique[PUCES].sum(axis=1) / ACTIONS).iloc[-1]]))

frais de reequilibrage entre poches : 2.659 pour 100 investis

poids cibles de la poche d'actions, en part des actions :
            acheteurs  vendeurs  electricite  equipement  immobilier  amont des puces  matieres et energie
2000-01-03      0.143     0.143        0.143       0.143       0.143            0.143                0.143
2001-01-02      0.080     0.077        0.200       0.203       0.133            0.068                0.239
2005-01-03      0.106     0.094        0.231       0.161       0.132            0.091                0.184
2010-01-04      0.148     0.131        0.225       0.143       0.099            0.120                0.136
2015-01-02      0.109     0.121        0.161       0.146       0.184            0.129                0.151
2020-01-02      0.136     0.102        0.202       0.137       0.171            0.101                0.152
2025-01-02      0.103     0.106        0.180       0.152       0.165            0.107                0.187
2026-01-02      0.122  

## 2. L'historique, à lire comme une vérification et non comme une preuve

P11 est comparé à `P1` rééquilibré, et au mélange de `P1` rééquilibré avec 40 % d'obligations intermédiaires de l'étape 4, qui a la même part d'actions.

In [3]:
table = pd.read_csv(SORTIE / "couverture_valeurs.csv.gz", index_col=0)
table.index = table.index.astype(str)
series = {"P11": p11, "P1_reeq": valeurs["P1_reeq"], "P1_reeq, 60 % et 40 % d'obligations": table["P1_reeq|treso_inter_60"],
          "P1_reeq, 60 % et 40 % de liquidites": table["P1_reeq|liquidites_60"]}

def mesures(v):
    r = v.pct_change().dropna()
    m = r_spy.reindex(r.index)
    ok = m.notna()
    d = risque.drawdown(v)
    return {"annualise": risque.annualiser_rendement(v), "vol_quotidienne": risque.volatilite(r),
            "vol_mensuelle": risque.volatilite(v.groupby(pd.Series(v.index).str[:7].values).last().pct_change().dropna(), risque.MOIS),
            "repli_max": d["repli_max"], "recuperation": d["recuperation"], "es99": risque.perte_au_dela(r, 0.99),
            "sharpe": risque.sharpe(r, sans_risque), "sortino": risque.sortino(r, sans_risque),
            "beta_spy": float(np.polyfit(m[ok], r[ok], 1)[0])}

tableau = pd.DataFrame({n: mesures(v) for n, v in series.items()}).T
print(tableau.round(4).to_string())
print()
AVANT, DEPUIS = slice(None, "2021-12-31"), slice("2022-01-01", None)
for nom, fenetre in [("avant 2022", AVANT), ("depuis 2022", DEPUIS)]:
    print("===", nom)
    print(pd.DataFrame({n: mesures(v.loc[fenetre]) for n, v in series.items()}).T
          [["annualise", "vol_quotidienne", "repli_max", "es99", "sharpe"]].round(4).to_string())
episodes = pd.read_csv(SORTIE / "episodes_tension.csv")
print()
print(pd.DataFrame({n: {e.debut[:4]: v.loc[e.creux] / v.loc[e.debut] - 1 for e in episodes.itertuples()}
                    for n, v in series.items()}).round(3).to_string())

                                     annualise  vol_quotidienne  vol_mensuelle  repli_max  recuperation    es99  sharpe  sortino  beta_spy
P11                                     0.1275           0.1212         0.1088    -0.3099         176.0  0.0276  0.8963   1.2836    0.5717
P1_reeq                                 0.1870           0.2173         0.1910    -0.5184         347.0  0.0510  0.8115   1.1626    1.0496
P1_reeq, 60 % et 40 % d'obligations     0.1343           0.1237         0.1132    -0.3035         153.0  0.0273  0.9286   1.3341    0.5760
P1_reeq, 60 % et 40 % de liquidites     0.1248           0.1282         0.1151    -0.3309         200.0  0.0288  0.8348   1.1907    0.6101

=== avant 2022
                                     annualise  vol_quotidienne  repli_max    es99  sharpe
P11                                     0.1259           0.1220    -0.3099  0.0284  0.9143
P1_reeq                                 0.1706           0.2179    -0.5184  0.0523  0.7656
P1_reeq, 60 % et

## 3. Ce que la règle devait corriger : la part des puces dans le risque

Contributions d'Euler des sept maillons au risque de la poche d'actions, sur la covariance des 252 dernières séances, aux poids de la dernière cible et aux poids égaux de `P1` rééquilibré par maillon, qui sont ceux du nombre d'entreprises.

In [4]:
recent = poches[list(MAILLONS.values())].iloc[-252:]
covariance = recent.cov().to_numpy() * 252
effectifs = pd.Series({nom: len(appartenance[appartenance.portefeuille == pf]) for pf, nom in MAILLONS.items()})
resultat = {}
for nom, w in [("P11, derniere cible", historique[list(MAILLONS.values())].iloc[-1] / ACTIONS),
               ("poids par nombre d'entreprises", effectifs / effectifs.sum())]:
    contributions, vol = risque.contributions_au_risque(w.to_numpy(), covariance)
    resultat[(nom, "poids")] = w
    resultat[(nom, "part du risque")] = pd.Series(contributions / vol, index=w.index)
    print(f"{nom} : volatilite de la poche d'actions {vol:.2%}, part des puces dans le risque "
          f"{(contributions / vol)[[list(w.index).index(p) for p in PUCES]].sum():.1%}")
print(pd.DataFrame(resultat).round(3).to_string())

P11, derniere cible : volatilite de la poche d'actions 15.31%, part des puces dans le risque 43.6%
poids par nombre d'entreprises : volatilite de la poche d'actions 21.30%, part des puces dans le risque 65.9%
                    P11, derniere cible                poids par nombre d'entreprises               
                                  poids part du risque                          poids part du risque
acheteurs                         0.122          0.085                          0.081          0.034
vendeurs                          0.089          0.212                          0.252          0.492
electricite                       0.210          0.094                          0.178          0.041
equipement                        0.136          0.184                          0.207          0.209
immobilier                        0.181          0.116                          0.052          0.016
amont des puces                   0.101          0.225                          0.09

## 4. Sorties et point de départ du test prospectif

In [5]:
pd.DataFrame({"P11": p11}).to_csv(SORTIE / "p11_valeurs.csv", encoding="utf-8")
historique.index.name = "date"
historique.to_csv(SORTIE / "p11_cibles.csv", encoding="utf-8")
tableau.to_csv(SORTIE / "p11_mesures.csv", encoding="utf-8")
depart = {"date": dates[-1], "P11": float(p11.iloc[-1]), "P1_reeq": float(valeurs["P1_reeq"].iloc[-1]),
          "cible_en_vigueur_depuis_2026-01-02": historique.iloc[-1].round(6).to_dict(),
          "predictions": ["vol quotidienne P11 <= 0,65 x vol P1_reeq sur 252 seances",
                          "part des puces dans le risque des actions de P11 <= 50 %",
                          "repli maximal P11 moins profond que celui de P1_reeq"],
          "sorties_sha256": {s: empreinte(SORTIE / s) for s in ["p11_valeurs.csv", "p11_cibles.csv", "p11_mesures.csv"]}}
(SORTIE / "p11_depart_prospectif.json").write_text(json.dumps(depart, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: v for k, v in depart.items() if k != "sorties_sha256"}, ensure_ascii=False, indent=1))

{
 "date": "2026-09-04",
 "P11": 2442.1390530045223,
 "P1_reeq": 9597.07209379358,
 "cible_en_vigueur_depuis_2026-01-02": {
  "acheteurs": 0.072994,
  "vendeurs": 0.053245,
  "electricite": 0.126019,
  "equipement": 0.081357,
  "immobilier": 0.108342,
  "amont des puces": 0.060747,
  "matieres et energie": 0.097296,
  "obligations": 0.2,
  "liquidites": 0.2
 },
 "predictions": [
  "vol quotidienne P11 <= 0,65 x vol P1_reeq sur 252 seances",
  "part des puces dans le risque des actions de P11 <= 50 %",
  "repli maximal P11 moins profond que celui de P1_reeq"
 ]
}
